# Qwen3.8-Flash-Next (180B, 6B chạy mỗi token) trên Colab G4: endpoint chuẩn OpenAI

**Run All** → dò máy (phải là G4, đo RAM và đĩa) → chọn bản nén theo RAM → cài vLLM vào môi trường riêng → tải model (133–186 GB) → chạy model, bảng n-gram để ở RAM máy → tự kiểm (trả lời, nghĩ, gọi tool, đọc ảnh, tốc độ) → mở gateway và tunnel → in khung **ENDPOINT SẴN SÀNG**.

| Bước | Việc cần làm |
|---|---|
| 1 | Runtime → Change runtime type → **G4** (bật **High-RAM** nếu có lựa chọn) |
| 2 | **Run All**, chờ khung `ENDPOINT SẴN SÀNG` ở ô 9 (lần đầu khoảng 20–30 phút: cài vLLM, tải model, nạp bảng n-gram vào RAM) |
| 3 | Chép `Base URL`, `API key`, `Model` vào app hay thư viện nói chuẩn OpenAI. Bấm Stop ô 10 không tắt endpoint; tắt hẳn: chạy `tat_het()` |

**Vì sao chạy được trên một thẻ 96 GB:** model có 180 tỷ tham số nhưng 51 tỷ trong đó là bảng tra n-gram, mỗi token chỉ đọc vài dòng của bảng. vLLM 0.31.0 tự để bảng này ở RAM máy và tra qua PCIe, còn GPU giữ phần thân (chuyên gia MoE nén NVFP4). Mỗi token chỉ chạy 6 tỷ tham số nên nhanh hơn Qwen3.8-27B dù tổng lớn hơn nhiều lần.

**Bản nén** (ô 2, `MODEL = "auto"` dùng `nvidia` khi đủ RAM và đĩa):

| Bản | Tải về | RAM cần cho bảng | Ghi chú |
|---|---|---|---|
| `nvidia` | 133 GB | khoảng 50 GB | bản chính chủ NVIDIA, bảng n-gram nén FP8 kèm hệ số; vLLM 0.31.0 có đường riêng cho kiểu nén này. NVIDIA chỉ thử trên B200/B300, chưa ai báo chạy trên RTX PRO 6000 |
| `primitive` | 186 GB | khoảng 100 GB | bảng n-gram BF16; nhóm làm bản này đã chạy trên một thẻ RTX PRO 6000 bằng ảnh Docker vLLM riêng (họ đo: 74 token/giây một người dùng, 484 token/giây 32 người). Config của bản này khai bảng là FP8 trong khi tệp là BF16: với vLLM 0.31.0 có thể hỏng lúc nạp, chưa ai thử |

**Giấy phép:** model gốc dùng Qwen Community License 1.0 (không phải Apache), bản NVIDIA thêm NVIDIA Open Model License. Đọc trước khi dùng cho việc thương mại: https://huggingface.co/Qwen/Qwen3.8-Flash-Next/blob/main/LICENSE

**Sơ đồ:** `127.0.0.1:8787 gateway → 127.0.0.1:8000 vLLM`. Notebook này mặc định **mở tunnel** (`TUNNEL = True` ở ô 2) để app ở máy nhà gọi vào. FAQ của Colab cấm "web service offerings not related to interactive compute" và "connecting to remote proxies"; tunnel có thuộc diện đó không thì Colab không nói rõ. Không muốn mở: `TUNNEL = False`.

**Secret** (biểu tượng chìa khóa bên trái, không bắt buộc):

| Secret | Tác dụng | Không đặt thì |
|---|---|---|
| `GATEWAY_TOKEN` | API key cố định | mỗi phiên sinh khóa mới, in ở khung |
| `CF_TUNNEL_TOKEN` + `PUBLIC_BASE_URL` | tunnel có tên của Cloudflare, địa chỉ cố định | Quick Tunnel, địa chỉ `trycloudflare.com` đổi mỗi phiên |
| `HF_TOKEN` | tải nhanh hơn, ít bị giới hạn lượt | tải ẩn danh |

In [ ]:
# 1) Dò máy: nền tảng, GPU, RAM, đĩa, mạng. Notebook này chỉ chạy trên G4 (Blackwell, compute capability 12.0).
import base64, hashlib, importlib.util, io, json, os, re, secrets, shutil, site, subprocess, sys, tarfile, time
import urllib.error, urllib.request
from pathlib import Path

NOTEBOOK_VERSION = "2026-10-13"

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ON_COLAB = not ON_KAGGLE and importlib.util.find_spec("google.colab") is not None
CHAY_NEN = ON_KAGGLE and os.environ["KAGGLE_KERNEL_RUN_TYPE"] != "Interactive"

WORK = Path("/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else "work").resolve()
SCRATCH = (Path("/tmp") if ON_KAGGLE else WORK) / "qwen38"
MODEL_DIR, BIN_DIR, LOG_DIR = SCRATCH / "models", SCRATCH / "bin", WORK / "qwen38_logs"
for _p in (MODEL_DIR, BIN_DIR, LOG_DIR):
    _p.mkdir(parents=True, exist_ok=True)


def nvidia_smi(*args):
    exe = shutil.which("nvidia-smi") or next(
        (p for p in ("/opt/bin/nvidia-smi", "/usr/bin/nvidia-smi", "/usr/local/nvidia/bin/nvidia-smi") if Path(p).exists()), None)
    if not exe:
        return ""
    r = subprocess.run([exe, *args], text=True, capture_output=True)
    return r.stdout.strip() if r.returncode == 0 else ""


def gpu_csv(fields):
    out = nvidia_smi(f"--query-gpu={fields}", "--format=csv,noheader,nounits")
    return [[x.strip() for x in line.split(",")] for line in out.splitlines() if line.strip()]


def ram_gib():
    """(tổng, còn dùng được) GiB, đọc /proc/meminfo."""
    m = dict(re.findall(r"^(\w+):\s+(\d+) kB", Path("/proc/meminfo").read_text(), re.M))
    return int(m["MemTotal"]) / 2**20, int(m["MemAvailable"]) / 2**20


GPUS = [{"index": int(i), "name": n, "gib": int(m) / 1024, "cc": cc, "driver": dv}
        for i, n, m, cc, dv in gpu_csv("index,name,memory.total,compute_cap,driver_version")]
if not GPUS or not GPUS[0]["cc"].startswith("12.") or GPUS[0]["gib"] < 90:
    raise SystemExit(
        f"Notebook này dành cho G4 (RTX PRO 6000 Blackwell 96 GB, compute capability 12.0). Máy này: "
        f"{[(g['name'], round(g['gib']), g['cc']) for g in GPUS] or 'không có GPU'}.\n"
        "  Colab: Runtime -> Change runtime type -> G4.")

try:
    urllib.request.urlopen("https://huggingface.co/api/models/Qwen/Qwen3.8-Flash-Next", timeout=20).read(64)
except urllib.error.HTTPError:
    pass
except Exception as e:
    raise SystemExit(f"Chưa có mạng ({type(e).__name__}: {e}).")

RAM_TONG, RAM_TRONG = ram_gib()
DIA_TRONG = shutil.disk_usage(SCRATCH).free / 1e9
print(json.dumps({
    "notebook": NOTEBOOK_VERSION, "kaggle": ON_KAGGLE, "colab": ON_COLAB,
    "python": sys.version.split()[0], "cpu": os.cpu_count(),
    "gpu": [f"{g['name']} {g['gib']:.1f} GiB, cc {g['cc']}, driver {g['driver']}" for g in GPUS],
    "ram_gib": f"{RAM_TRONG:.0f} trống / {RAM_TONG:.0f}", "dia_trong_gb": round(DIA_TRONG, 1),
}, ensure_ascii=False, indent=1))

In [ ]:
# 2) CẤU HÌNH: chỉ cần sửa ô này.
MODEL = "auto"           # "auto" = "nvidia" khi đủ RAM và đĩa | "nvidia" | "primitive" (bảng ở ô chữ đầu)
SPEC = "mtp"             # đoán trước bằng đầu MTP có sẵn trong model | "tat"
SPEC_N = 3               # số token đoán mỗi lượt (Qwen ghi 3 trong hướng dẫn)
CTX = "auto"             # "auto" = 131072; 262144 cùng MTP không vừa 96 GB (primitive-ai đo: KV cần 7,57 GiB, còn 5,95)
KV = "auto"              # bộ nhớ ngữ cảnh: "auto" (bf16, cách đã có người chạy trên chip này) | "fp8" (vLLM 0.31.0 mới thêm, chưa ai thử ở đây)
VISION = True            # nạp phần đọc ảnh
THINKING = True          # mặc định có nghĩ; từng lượt gọi vẫn tự bật tắt được
MAX_SEQS = 4             # số lượt hỏi chạy song song tối đa
GPU_UTIL = 0.92          # phần VRAM vLLM được giữ (primitive-ai chạy 0.92: dùng 88,8 GiB)
VLLM_THEM = []           # cờ vLLM thêm. Thử ["--moe-backend", "marlin"]: primitive-ai đo một người dùng nhanh hơn 10% (93,2 so với
                         # 84,6 token/giây, không MTP), 32 người chậm hơn 1–3%
TUNNEL = True            # mở tunnel để gọi endpoint từ máy khác (lưu ý FAQ Colab in ở ô 9
                         # khi tắt) | "auto": Colab không mở | False
KEEP_ALIVE_HOURS = 4     # ô 10 giữ phiên bấy nhiêu giờ rồi tự tắt (G4 tốn đơn vị tính toán nhanh)

VLLM_VERSION = "0.31.0"  # 05/10/2026: có Qwen3.8-Flash-Next (Qwen4Exp), bảng n-gram tự để ở RAM máy, sửa MTP + backend MoE (#54788)
VLLM_SHA256 = "39132b433967f0962d03d8a5f633b9bee6b32708f566c7d5637b79f8d0ef3433"     # bánh xe x86_64, PyPI 13/10/2026
VLLM_PIN_DATE = "2026-10-13T00:00:00Z"
VLLM_CHO_GIAY = 3600     # chờ nạp tối đa (đọc khoảng 90 GB lên GPU và đến 95 GB bảng vào RAM)
# Nhân FlashInfer dựng sẵn cho Blackwell (SM120), cùng bản 0.7.0.post1 mà vLLM 0.31.0 ghim; địa chỉ và mã băm đọc từ
# https://flashinfer.ai/whl/cu130/ ngày 13/10/2026 (gói vỏ 5 KB + gói nhân 187 MB). Thiếu thì vLLM tự dịch nhân lúc nạp.
_FI = "https://github.com/flashinfer-ai/flashinfer/releases/download/v0.7.0.post1/flashinfer_jit_cache"
FI_CACHE = [(f"{_FI}-0.7.0.post1+cu130-cp39-abi3-manylinux_2_28_x86_64.whl",
             "9630eeb32396adb89ce8ffa8465fedd14549eb2e2ad3aff182d9e69be4a12075"),
            (f"{_FI}_sm120f-0.7.0.post1+cu130-cp39-abi3-manylinux_2_28_x86_64.whl",
             "1c8b5a3614fdc9ad4f0e26502c5438657e28d667d165b05ff0383ca8d208568f")]
ENGINE_PORT = 8000       # vLLM, chỉ nghe 127.0.0.1
PORT_GW = 8787           # gateway, cổng DUY NHẤT đưa ra tunnel

# (repo, GB tải về, GiB RAM trống cần cho bảng n-gram + phần còn lại của tiến trình). Cỡ đọc từ API Hugging Face 13/10/2026.
# Cả hai giữ đầu MTP và phần đọc ảnh; không có code chạy kèm (vLLM có sẵn kiến trúc này, không bật trust-remote-code).
MODELS = {
    "nvidia": ("nvidia/Qwen3.8-Flash-Next-NVFP4", 132.7, 62),             # bảng FP8 khoảng 47 GiB (modelopt MIXED_PRECISION)
    "primitive": ("primitive-ai/Qwen3.8-Flash-Next-NVFP4", 186.4, 110),   # bảng BF16 95,4 GB
}
DRAFTS = {}              # model này đoán trước bằng đầu MTP của chính nó
ALIAS = "qwen3.8-flash-next"
if MODEL == "auto":
    _r, _gb, _ram = MODELS["nvidia"]
    MODEL = "nvidia" if RAM_TRONG >= _ram and DIA_TRONG >= _gb + 8 else None
    if MODEL is None:
        raise SystemExit(
            f"Máy này còn {RAM_TRONG:.0f} GiB RAM và {DIA_TRONG:.0f} GB đĩa trống. Cần ít nhất "
            f"{MODELS['nvidia'][2]} GiB RAM (giữ bảng n-gram) và {MODELS['nvidia'][1] + 8:.0f} GB đĩa (tải model), bản nvidia.\n"
            "  Colab: Runtime -> Change runtime type -> G4, bật High-RAM nếu có; hoặc dùng notebook 'Qwen3.8-27B G4'.")
MODEL_REPO, MODEL_GB, _RAM_CAN = MODELS[MODEL]
if RAM_TRONG < _RAM_CAN:
    raise SystemExit(f"Bản '{MODEL}' cần khoảng {_RAM_CAN} GiB RAM trống, máy này còn {RAM_TRONG:.0f}. Đặt MODEL = \"auto\".")
SPEC_REPO = None
SPEC_SO = int(SPEC_N)
CTX_SIZE = 131072 if CTX == "auto" else int(CTX)
print(json.dumps({"model": MODEL_REPO, "doan_truoc": f"{SPEC} {SPEC_SO if SPEC != 'tat' else ''}".strip(), "ngu_canh": CTX_SIZE,
                  "kv": KV, "doc_anh": VISION, "nghi_mac_dinh": THINKING, "vllm": VLLM_VERSION}, ensure_ascii=False, indent=1))

In [ ]:
# 3) Khóa bí mật (không bắt buộc), thư viện cho gateway, hàm dùng chung
def get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name) or None
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name) or None
    except Exception:
        return None
    return None


HF_TOKEN = get_secret("HF_TOKEN")
GW_TOKEN_CO_DINH = get_secret("GATEWAY_TOKEN")
CF_TOKEN = get_secret("CF_TUNNEL_TOKEN")
PUBLIC_BASE_URL = (get_secret("PUBLIC_BASE_URL") or "").rstrip("/")
API_KEY = "sk-" + secrets.token_hex(24)          # khóa nội bộ gateway -> máy chạy model, không ra ngoài
print("HF_TOKEN:", "có" if HF_TOKEN else "không (đủ cho repo công khai)")
print("GATEWAY_TOKEN:", "lấy từ Secret (cố định)" if GW_TOKEN_CO_DINH else "không có, sẽ sinh khóa mới cho phiên này")
print("CF_TUNNEL_TOKEN:", "có (tunnel có tên)" if CF_TOKEN else "không có (Quick Tunnel, địa chỉ đổi mỗi phiên)")

_thieu = [m for m in ("fastapi", "uvicorn", "httpx", "requests", "huggingface_hub") if importlib.util.find_spec(m) is None]
if _thieu:
    print("Cài thêm:", _thieu)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi>=0.116,<1", "uvicorn>=0.35,<1",
                    "httpx>=0.28,<1", "requests>=2.31", "huggingface_hub>=0.28"], check=True)
import requests

KHONG_NGHI = {"chat_template_kwargs": {"enable_thinking": False}}


def tai(url, dst, sha256=None):
    """Tải url về dst (ghi file tạm rồi đổi tên); có sha256 thì kiểm."""
    dst = Path(dst)
    tmp = dst.with_name(dst.name + ".part")
    h, n, t0 = hashlib.sha256(), 0, time.time()
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=120) as r, \
            open(tmp, "wb") as f:
        while True:
            b = r.read(4 << 20)
            if not b:
                break
            f.write(b)
            h.update(b)
            n += len(b)
    if sha256 and h.hexdigest() != sha256:
        tmp.unlink()
        raise RuntimeError(f"Sai mã băm, không dùng file này: {dst.name}")
    tmp.replace(dst)
    print(f"  tải {dst.name}: {n / 2**20:.0f} MiB trong {time.time() - t0:.0f} giây"
          + (", sha256 khớp" if sha256 else ", KHÔNG có mã băm để kiểm"))
    return dst


def chay_nen(lenh, **kw):
    """subprocess.Popen cho tiến trình chạy lâu (máy chạy model, gateway, tunnel). Nhóm tiến trình riêng: bấm Stop một ô thì
    Jupyter gửi SIGINT cho cả nhóm của kernel (13/10 Colab G4: Stop ô 10 tắt luôn vLLM). PR_SET_PDEATHSIG: kernel chết
    (Restart session) thì tiến trình nhận SIGTERM, không sót lại giữ GPU và RAM."""
    import ctypes, signal
    prctl, sigterm = ctypes.CDLL("libc.so.6", use_errno=True).prctl, int(signal.SIGTERM)   # nạp sẵn ở tiến trình cha:
                                                                                          # trong con chỉ gọi hàm, không nạp gì

    def _theo_kernel():
        prctl(1, sigterm)                                                     # 1 = PR_SET_PDEATHSIG

    return subprocess.Popen(lenh, start_new_session=True, preexec_fn=_theo_kernel, **kw)


def hoi(messages, url=None, headers=None, timeout=900, **kw):
    """Một lượt chat không stream. Mặc định gọi thẳng máy chạy model trong máy (không qua gateway)."""
    r = requests.post(url or f"http://127.0.0.1:{ENGINE_PORT}/v1/chat/completions",
                      headers=headers or {"Authorization": f"Bearer {API_KEY}"},
                      json={"model": ALIAS, "messages": messages, **kw}, timeout=timeout)
    r.raise_for_status()
    return r.json()


def anh_7391():
    """Ảnh PNG vẽ số 7391, dạng data URL, để thử phần đọc ảnh."""
    from PIL import Image, ImageDraw, ImageFont
    img = Image.new("RGB", (448, 224), "white")
    try:
        font = ImageFont.load_default(size=110)
    except TypeError:
        font = ImageFont.load_default()
    ImageDraw.Draw(img).text((70, 50), "7391", fill="black", font=font)
    buf = io.BytesIO()
    img.save(buf, "PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def hoi_anh(timeout=900):
    """Hỏi số trong ảnh 7391 qua API nội bộ, trả chữ model đáp."""
    j = hoi([{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": anh_7391()}},
        {"type": "text", "text": "Trong ảnh có số nào? Chỉ trả lời bằng chữ số."}]}],
        timeout=timeout, max_tokens=32, temperature=0, **KHONG_NGHI)
    return (j["choices"][0]["message"].get("content") or "").strip()

In [ ]:
# 4) vLLM: cài vào môi trường Python riêng (không đụng PyTorch có sẵn của Colab), kiểm mã băm, kiểm thấy GPU
VENV = SCRATCH / "vllm-venv"
VPY = VENV / "bin" / "python"


def chay(cmd, **kw):
    """Chạy lệnh, in 3.000 ký tự cuối khi hỏng; trả mã thoát."""
    r = subprocess.run([str(c) for c in cmd], text=True, capture_output=True, **kw)
    if r.returncode:
        print(f"  lệnh hỏng (mã {r.returncode}): {' '.join(map(str, cmd))[:200]}\n" + ((r.stdout or "") + (r.stderr or ""))[-3000:])
    return r.returncode


def vllm_ok():
    r = subprocess.run([str(VPY), "-c", "import torch, vllm; print(vllm.__version__, torch.__version__, torch.version.cuda, "
                        "torch.cuda.is_available(), torch.cuda.get_device_capability() if torch.cuda.is_available() else None)"],
                       text=True, capture_output=True)
    return r.returncode == 0 and f"{VLLM_VERSION} " in r.stdout and " True " in r.stdout, (r.stdout + r.stderr).strip()[-1500:]


t0 = time.time()
if not shutil.which("uv"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv") or str(Path(sys.executable).parent / "uv")
ok, chu = vllm_ok() if VPY.exists() else (False, "")
if not ok:
    shutil.rmtree(VENV, ignore_errors=True)
    if chay([UV, "venv", VENV, "--python", sys.executable, "-q"]):
        raise SystemExit("Không tạo được môi trường Python riêng cho vLLM.")
    # Lấy địa chỉ bánh xe từ PyPI rồi kiểm mã băm đã ghim ở ô 2.
    _ten = f"vllm-{VLLM_VERSION}-cp38-abi3-manylinux_2_28_x86_64.whl"
    _pypi = json.load(urllib.request.urlopen(f"https://pypi.org/pypi/vllm/{VLLM_VERSION}/json", timeout=60))
    _url = next(f["url"] for f in _pypi["urls"] if f["filename"] == _ten)
    _whl = tai(_url, SCRATCH / _ten, VLLM_SHA256)
    # Thư viện đi kèm (PyTorch, FlashInfer...) lấy đúng bản PyPI có tới ngày ghim, để lần chạy sau ra cùng một bộ.
    if chay([UV, "pip", "install", "--python", VPY, "--exclude-newer", VLLM_PIN_DATE, _whl], env=dict(os.environ, UV_HTTP_TIMEOUT="300")):
        raise SystemExit("Cài vLLM hỏng (xem chữ in ở trên).")
    _whl.unlink(missing_ok=True)
    ok, chu = vllm_ok()
# Trên chip mới vLLM dịch nhân FlashInfer ngay lúc nạp và cần lệnh ninja (Colab G4 13/10: "No such file or directory: 'ninja'").
# Ô 6 đưa thư mục bin của môi trường này lên đầu PATH để tìm thấy nó.
if ok and not (VENV / "bin" / "ninja").exists() and chay([UV, "pip", "install", "--python", VPY, "--exclude-newer", VLLM_PIN_DATE, "ninja"]):
    raise SystemExit("Cài ninja hỏng (xem chữ in ở trên).")
# Nhân FlashInfer dựng sẵn cho đúng chip (FI_CACHE ở ô 2): có thì vLLM nạp thẳng, không phải dịch lúc nạp (cần nvcc, thêm vài
# phút mỗi phiên). Cài --no-deps vì gói vỏ kéo theo bản dựng sẵn của mọi loại chip. Hỏng thì bỏ qua, vLLM tự dịch như cũ.
_fi_dau = VENV / ".fi-cache"
if ok and globals().get("FI_CACHE") and not _fi_dau.exists():
    try:
        _whls = [tai(u, SCRATCH / u.rsplit("/", 1)[1], s) for u, s in FI_CACHE]
        if chay([UV, "pip", "install", "--python", VPY, "--no-deps", *_whls]) == 0:
            _fi_dau.write_text("ok")
        for _w in _whls:
            _w.unlink(missing_ok=True)
    except Exception as e:
        print(f"  (không cài được nhân FlashInfer dựng sẵn: {type(e).__name__}: {str(e)[:200]}; vLLM sẽ tự dịch lúc nạp)")
if not ok:
    raise SystemExit("vLLM cài xong nhưng không thấy GPU (thường là driver quá cũ cho CUDA 13). Chữ in ra:\n" + chu
                     + f"\nDriver máy này: {nvidia_smi('--query-gpu=driver_version', '--format=csv,noheader')}")
print(f"[OK] vLLM sẵn sàng sau {time.time() - t0:.0f} giây: {chu}")

In [ ]:
# 5) Model và bộ đoán trước: tải từ Hugging Face về SCRATCH (đã có thì bỏ qua)
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
if CHAY_NEN:
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import snapshot_download


def lay_repo(repo):
    dst = MODEL_DIR / repo.replace("/", "__")
    t0 = time.time()
    p = Path(snapshot_download(repo, local_dir=str(dst), token=HF_TOKEN or None, max_workers=16))
    gb = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9
    print(f"  {repo}: {gb:.2f} GB, {time.time() - t0:.0f} giây")
    return p


trong = shutil.disk_usage(MODEL_DIR).free / 1e9
if trong < MODEL_GB + 8:
    raise SystemExit(f"Đĩa {MODEL_DIR} còn {trong:.0f} GB, không đủ cho model khoảng {MODEL_GB:.0f} GB.")
model_path = lay_repo(MODEL_REPO)
spec_path = lay_repo(SPEC_REPO) if SPEC_REPO else None
print("[OK] Model:", model_path)

In [ ]:
# 6) Chạy vLLM (chỉ nghe 127.0.0.1) và chờ nạp xong. Hỏng thì lùi dần: bộ đoán riêng (DSpark/DFlash2) -> MTP -> không đoán trước;
#    KV fp8 -> bf16;
#    hết bộ nhớ thì nửa ngữ cảnh. Lần đầu mất vài phút để dịch đồ thị (torch.compile, CUDA graph).
VLLM_ENV = dict(os.environ, HF_HUB_OFFLINE="1", VLLM_CACHE_ROOT=str(SCRATCH / "vllm-cache"),
                TORCHINDUCTOR_CACHE_DIR=str(SCRATCH / "inductor-cache"), VLLM_LOGGING_LEVEL="INFO",
                # ninja của môi trường vLLM và nvcc của máy, cho phần dịch nhân lúc nạp (xem ô 4)
                PATH=":".join([str(VPY.parent), *[d for d in ("/usr/local/cuda/bin",) if Path(d).is_dir()], os.environ.get("PATH", "")]),
                **globals().get("VLLM_ENV_THEM", {}))


def spec_json(spec):
    if spec in DRAFTS:
        return json.dumps({"model": str(spec_path), "method": DRAFTS[spec][1], "num_speculative_tokens": SPEC_SO})
    if spec == "mtp":
        return json.dumps({"method": "mtp", "num_speculative_tokens": SPEC_SO if SPEC == "mtp" else 3})
    return None


def lenh_vllm(spec, ctx, kv):
    a = [str(VPY), "-m", "vllm.entrypoints.openai.api_server", "--model", str(model_path), "--served-model-name", ALIAS,
         "--host", "127.0.0.1", "--port", str(ENGINE_PORT), "--api-key", API_KEY,
         "--max-model-len", str(ctx), "--max-num-seqs", str(MAX_SEQS), "--gpu-memory-utilization", str(GPU_UTIL),
         "--kv-cache-dtype", kv, "--reasoning-parser", "qwen3", "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
         "--disable-uvicorn-access-log", *VLLM_THEM]
    if spec_json(spec):
        a += ["--speculative-config", spec_json(spec)]
    if not VISION:
        a += ["--limit-mm-per-prompt", json.dumps({"image": 0, "video": 0})]
    if not THINKING:
        a += ["--default-chat-template-kwargs", json.dumps({"enable_thinking": False})]
    return a


def cho_song(proc, timeout_s):
    t0, cuoi = time.time(), None
    while time.time() - t0 < timeout_s:
        if proc.poll() is not None:
            return False, f"tiến trình thoát với mã {proc.returncode}"
        try:
            r = requests.get(f"http://127.0.0.1:{ENGINE_PORT}/health", timeout=3)
            if r.status_code == 200:
                return True, None
            cuoi = r.status_code
        except Exception as e:
            cuoi = type(e).__name__
        time.sleep(5)
    return False, f"quá {timeout_s} giây chưa lên ({cuoi})"


def DEM_MTP():
    """(số token đoán trúng, số token đã đoán) cộng dồn, đọc từ /metrics của vLLM."""
    try:
        t = requests.get(f"http://127.0.0.1:{ENGINE_PORT}/metrics", timeout=5).text
    except Exception:
        return None
    lay = lambda ten: sum(float(m) for m in re.findall(rf"^vllm:{ten}(?:{{[^}}]*}})? ([0-9.e+]+)$", t, re.M))
    return lay("spec_decode_num_accepted_tokens_total"), lay("spec_decode_num_draft_tokens_total")


if globals().get("ENGINE_PROC") is not None and ENGINE_PROC.poll() is None:     # chạy lại ô này: tắt bản cũ trước
    ENGINE_PROC.terminate()
    ENGINE_PROC.wait(timeout=120)

_specs = ["tat"] if SPEC == "tat" else list(dict.fromkeys([SPEC, "mtp", "tat"]))     # đã tắt thì không tự bật lại
_kvs = [KV] + (["auto"] if KV != "auto" else [])
_si = _ki = 0
_ctx = CTX_SIZE
ENGINE_TEN, ENGINE_LOG = f"vLLM {VLLM_VERSION}", LOG_DIR / "vllm.log"
for _lan in range(6):
    SPEC_DUNG, KV_DUNG = _specs[_si], _kvs[_ki]
    lenh = lenh_vllm(SPEC_DUNG, _ctx, KV_DUNG)
    print("Lệnh:", " ".join(a if a != API_KEY else "<khóa nội bộ>" for a in lenh))
    _f = open(ENGINE_LOG, "w")
    ENGINE_PROC = chay_nen(lenh, stdout=_f, stderr=subprocess.STDOUT, env=VLLM_ENV)
    t0 = time.time()
    ok, cuoi = cho_song(ENGINE_PROC, timeout_s=globals().get("VLLM_CHO_GIAY", 2400))
    if ok:
        CTX_SIZE = _ctx
        break
    if ENGINE_PROC.poll() is None:
        ENGINE_PROC.kill()
        ENGINE_PROC.wait(timeout=120)
    _f.close()
    _log = ENGINE_LOG.read_text(errors="ignore")
    _loi = [l for l in _log.splitlines() if re.search(r"Error|error:|out of memory|not supported|RESOURCE_EXHAUSTED", l)]
    print(f"  không nạp được (đoán trước {SPEC_DUNG}, ngữ cảnh {_ctx}, kv {KV_DUNG}): {cuoi}\n  " + "\n  ".join(l[:220] for l in _loi[-6:]))
    # Hết bộ nhớ: giữ cách đoán trước, giảm ngữ cảnh (về đúng số vLLM ước được nếu nó in ra, không thì một nửa).
    # Lỗi về kiểu KV: bỏ fp8. Lỗi khác: lùi một bậc đoán trước.
    _uoc = re.findall(r"estimated maximum model length is (\d+)", _log)
    if re.search(r"out of memory|No available memory for the cache blocks|larger than the maximum number of tokens that can be stored"
                 r"|KV cache is needed|RESOURCE_EXHAUSTED|Ran out of memory", _log, re.I) and min(
                     _ctx // 2, int(_uoc[-1]) if _uoc else _ctx) >= 16384:
        _ctx = int(_uoc[-1]) // 4096 * 4096 if _uoc and int(_uoc[-1]) < _ctx else _ctx // 2
    elif _ki + 1 < len(_kvs) and re.search(r"fp8|kv_cache_dtype|kv-cache-dtype", "\n".join(_loi), re.I):
        _ki += 1
    elif _si + 1 < len(_specs):
        _si += 1
    elif _ki + 1 < len(_kvs):
        _ki += 1
    else:
        break
if not ok:
    raise SystemExit("vLLM không lên. Cuối log:\n" + ENGINE_LOG.read_text(errors="ignore")[-5000:])
SPEC_TEN = {"dspark": "DSpark", "dflash2": "DFlash2", "mtp": "MTP"}.get(SPEC_DUNG)
DOC_ANH, MTP_TOKENS = VISION, (json.loads(spec_json(SPEC_DUNG))["num_speculative_tokens"] if SPEC_TEN else 0)
print(f"[OK] vLLM nạp xong sau {time.time() - t0:.0f} giây (PID {ENGINE_PROC.pid}, ngữ cảnh {CTX_SIZE}, kv {KV_DUNG}, "
      f"đoán trước {SPEC_TEN or 'tắt'}{f' {MTP_TOKENS} token' if SPEC_TEN else ''})")
for _dong in re.findall(r"[^\n]*(?:Model loading took|Available KV cache memory|GPU KV cache size|Maximum concurrency|"
                        r"Graph capturing finished|init engine)[^\n]*", ENGINE_LOG.read_text(errors="ignore"))[:8]:
    print("  ", _dong.strip()[-170:])
for i, used, total in gpu_csv("index,memory.used,memory.total"):
    print(f"   GPU {i}: {used}/{total} MiB")

In [ ]:
# 7) Tự kiểm qua API: trả lời, tốc độ, nghĩ + stream, gọi tool, đọc ảnh. Lỗi ở hai mục đầu thì dừng.
LOCAL = f"http://127.0.0.1:{ENGINE_PORT}/v1/chat/completions"
H_LOCAL = {"Authorization": f"Bearer {API_KEY}"}
KET_QUA = {}


def do_toc_do(de, max_tokens=300):
    """Đo tốc độ sinh của một lượt stream (token/giây, không tính đoạn đọc đề). Trả (số, chữ mô tả).
    llama-server tự đo (trường timings) thì lấy số đó; máy khác (vLLM) thì tính ở đây:
    (số token - 1) / (giờ nhận token cuối - giờ nhận token đầu)."""
    t_dau = t_cuoi = None
    n_khuc, usage, tm = 0, {}, {}
    mtp0 = DEM_MTP() if globals().get("DEM_MTP") else None
    with requests.post(LOCAL, headers=H_LOCAL, stream=True, timeout=900, json={
            "model": ALIAS, "stream": True, "stream_options": {"include_usage": True}, "max_tokens": max_tokens,
            "temperature": 0.7, "messages": [{"role": "user", "content": de}], **KHONG_NGHI}) as s:
        s.raise_for_status()
        for line in s.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:") or "[DONE]" in line:
                continue
            try:
                c = json.loads(line[5:])
            except Exception:
                continue
            usage, tm = c.get("usage") or usage, c.get("timings") or tm
            if any((ch.get("delta") or {}).get("content") for ch in c.get("choices") or []):
                t_cuoi = time.time()
                t_dau = t_dau or t_cuoi
                n_khuc += 1
    n = usage.get("completion_tokens") or n_khuc
    tinh = (n - 1) / (t_cuoi - t_dau) if t_dau and t_cuoi > t_dau and n > 1 else None
    so = tm.get("predicted_per_second") or tinh
    trung, ten_doan = "", globals().get("SPEC_TEN") or "MTP"     # SPEC_TEN: ô 6 vLLM đặt (DSpark, DFlash2, MTP)
    if tm.get("draft_n"):
        trung = f", {ten_doan} đoán trúng {tm['draft_n_accepted']}/{tm['draft_n']}"
    elif mtp0 is not None:
        mtp1 = DEM_MTP()
        if mtp1 and mtp1[1] > mtp0[1]:
            trung = f", {ten_doan} đoán trúng {mtp1[0] - mtp0[0]:.0f}/{mtp1[1] - mtp0[1]:.0f}"
    return so, (f"{so:.1f} token/giây ({n} token{trung})" if so else "không đo được")


# (a) trả lời đúng một chuỗi, không nghĩ
j = hoi([{"role": "user", "content": "Reply with exactly: QWEN_OK"}], max_tokens=32, temperature=0, **KHONG_NGHI)
_txt = (j["choices"][0]["message"].get("content") or "").strip()
assert "QWEN_OK" in _txt, f"Trả lời sai: {_txt!r} | {json.dumps(j)[:400]}"
KET_QUA["trả lời (không nghĩ)"] = f"đạt: {_txt!r}"

# (b) tốc độ sinh chữ: một đoạn code và một đoạn văn tiếng Việt. Lượt sinh dài đầu tiên sau khi nạp còn gánh phần khởi động
# của đường đoán trước (13/10 FableCoder MTP 2: lượt code đo đầu tiên 27,6 token/giây, bài đo có làm nóng 93,0): chạy một lượt nháp.
# Có lượt nháp rồi vẫn có lúc lượt code đầu bị khựng (13/10 13:46: 48,1 trong khi văn đo liền sau 75,0, các lượt khác 91–94):
# đo code hai lần, lấy lượt nhanh hơn, và in cả lượt đầu khi nó chậm hơn từ 20%.
do_toc_do("Đếm từ 1 đến 40, mỗi số một dòng.", max_tokens=96)
_hai_lan = [do_toc_do("Viết lớp Python HangDoi (hàng đợi ưu tiên dùng heapq) có các hàm them, lay, xem, kèm docstring. Chỉ trả code.")
            for _ in range(2)]
TOC_DO, _code = max(_hai_lan, key=lambda x: x[0] or 0)
if _hai_lan[0][0] and TOC_DO and _hai_lan[0][0] < 0.8 * TOC_DO:
    _code += f", lượt đầu chỉ {_hai_lan[0][0]:.1f}"
_, _van = do_toc_do("Viết một đoạn văn khoảng 200 từ giới thiệu Việt Nam.")
KET_QUA["tốc độ sinh"] = f"code {_code} | văn {_van}"

# (c) có nghĩ + stream, mức nghĩ "low". Bật nghĩ rõ ràng:
# model Qwen nhỏ mặc định không nghĩ (13/10 bài thử Qwen3.5-2B trả "85", phần nghĩ 0 ký tự)
try:
    _nghi, _noi = [], []
    with requests.post(LOCAL, headers=H_LOCAL, stream=True, timeout=900, json={
            "model": ALIAS, "stream": True, "max_tokens": 4096,
            "messages": [{"role": "user", "content": "17 nhân 23 bằng bao nhiêu? Trả lời một con số."}],
            "chat_template_kwargs": {"enable_thinking": True, "reasoning_effort": "low"}}) as s:
        s.raise_for_status()
        for line in s.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data:") or "[DONE]" in line:
                continue
            try:
                d = (json.loads(line[5:]).get("choices") or [{}])[0].get("delta", {})
            except Exception:
                continue
            _nghi.append(d.get("reasoning_content") or d.get("reasoning") or "")
            _noi.append(d.get("content") or "")
    _nghi, _noi = "".join(_nghi), "".join(_noi)
    KET_QUA["nghĩ + stream"] = (f"đạt: trả {_noi.strip()[:40]!r}, phần nghĩ {len(_nghi)} ký tự tách riêng" if "391" in _noi
                                else f"SAI: trả {_noi.strip()[:80]!r} (đúng là 391), phần nghĩ {len(_nghi)} ký tự")
except Exception as e:
    KET_QUA["nghĩ + stream"] = f"LỖI {type(e).__name__}: {str(e)[:160]}"

# (d) gọi tool
try:
    j = hoi([{"role": "user", "content": "Thời tiết ở Hà Nội bây giờ thế nào? Hãy gọi công cụ."}], max_tokens=512, **KHONG_NGHI,
            tools=[{"type": "function", "function": {
                "name": "get_weather", "description": "Lấy thời tiết hiện tại của một thành phố",
                "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}])
    _tc = j["choices"][0]["message"].get("tool_calls") or []
    KET_QUA["gọi tool"] = (f"đạt: {_tc[0]['function']['name']}({_tc[0]['function']['arguments']})" if _tc
                           else f"KHÔNG gọi tool, trả chữ: {(j['choices'][0]['message'].get('content') or '')[:80]!r}")
except Exception as e:
    KET_QUA["gọi tool"] = f"LỖI {type(e).__name__}: {str(e)[:160]}"

# (e) đọc ảnh: ảnh vẽ số 7391
if DOC_ANH:
    try:
        _txt = hoi_anh()
        KET_QUA["đọc ảnh"] = f"đạt: đọc ra {_txt!r}" if "7391" in _txt else f"SAI: đọc ra {_txt!r} (đúng là 7391)"
    except Exception as e:
        KET_QUA["đọc ảnh"] = f"LỖI {type(e).__name__}: {str(e)[:160]}"
else:
    KET_QUA["đọc ảnh"] = "tắt (VISION = False)"

for k, v in KET_QUA.items():
    print(f"  {k:<22} {v}")
if ENGINE_PROC.poll() is not None:
    raise SystemExit(f"{ENGINE_TEN} đã chết trong lúc tự kiểm (mã {ENGINE_PROC.returncode}). Cuối log:\n"
                     + ENGINE_LOG.read_text(errors="ignore")[-3000:]
                     + "\nThấy 'out of memory': hạ CTX (ví dụ 65536) hoặc đặt VISION = False ở ô 2, rồi chạy lại từ ô 2.")
print("[OK] Model trả lời đúng qua API nội bộ.")

In [ ]:
%%writefile gateway.py
"""Narrow OpenAI-compatible inference gateway. Tunnel THIS port (8787), never llama-server (8080) directly.

Security Invariants:
1. Strict allowlist: ONLY /healthz, /capabilities, /v1/models, /v1/chat/completions.
2. Distinct public GATEWAY_TOKEN vs internal VLLM_API_KEY.
3. Unknown routes return 404 (no catch-all proxy).
4. Unbuffered SSE streaming pass-through.
5. Strict request body limit (25 MB default).
6. Personal single-client concurrency guard (semaphore).
7. Zero internal credential or traceback leakage.
8. Request ID correlation logging without prompt content exposure.
9. Clean async lifespan management and shutdown.
"""
import asyncio
from contextlib import asynccontextmanager
import json
import logging
import os
import secrets
import time
from typing import AsyncGenerator

from fastapi import FastAPI, Header, HTTPException, Request, Response
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse
import httpx

# Logging configuration (Request ID and metadata only, NO prompt/body contents)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("qwen_gateway")

# Configuration & secrets
VLLM_URL = os.getenv("VLLM_URL", "http://127.0.0.1:8080").rstrip("/")
GATEWAY_TOKEN = os.getenv("GATEWAY_TOKEN", "")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "")
MODEL = os.getenv("SERVED_MODEL_NAME", "qwen3.8-27b")
MAX_MODEL_LEN = int(os.getenv("MAX_MODEL_LEN", "131072"))
MULTIMODAL = os.getenv("TEXT_ONLY", "false").lower() not in {"1", "true", "yes"}
MTP_TOKENS = int(os.getenv("MTP_TOKENS", "3"))
RUN_ID = os.getenv("RUN_ID", str(int(time.time())))
MAX_BODY_BYTES = int(os.getenv("GATEWAY_MAX_BODY_BYTES", str(25 * 1024 * 1024))) # 25 MB
MAX_CONCURRENT_INFERENCE = int(os.getenv("GATEWAY_CONCURRENCY", "3"))
MAX_OUTPUT_TOKENS = int(os.getenv("GATEWAY_MAX_OUTPUT_TOKENS", "32768"))
NONSTREAM_WAIT_S = float(os.getenv("GATEWAY_NONSTREAM_WAIT_S", "60"))  # below Cloudflare's 100 s cut-off
NONSTREAM_PAD_S = 15.0

if not GATEWAY_TOKEN or not VLLM_API_KEY:
    raise RuntimeError("Both GATEWAY_TOKEN and VLLM_API_KEY environment variables are required")

# State & Lifespan
client_holder: dict[str, httpx.AsyncClient] = {}
concurrency_limit = asyncio.Semaphore(MAX_CONCURRENT_INFERENCE)

@asynccontextmanager
async def lifespan(app: FastAPI):
    limits = httpx.Limits(max_keepalive_connections=10, max_connections=20)
    timeout = httpx.Timeout(900.0, connect=10.0, read=900.0, write=30.0)
    client = httpx.AsyncClient(limits=limits, timeout=timeout)
    client_holder["client"] = client
    logger.info(f"Gateway initialized. Target upstream at {VLLM_URL}, Model: {MODEL}")
    try:
        yield
    finally:
        logger.info("Gateway shutting down, closing upstream connections...")
        await client.aclose()
        logger.info("Gateway shutdown complete.")

app = FastAPI(
    docs_url=None,
    redoc_url=None,
    openapi_url=None,
    lifespan=lifespan,
)

# CORS for browser clients (`npm run dev` on localhost calling the tunnel URL).
# Auth is NOT weakened: preflight OPTIONS carries no credentials, and every
# allowlisted route still requires the Bearer gateway token.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type"],
    max_age=600,
)

@app.exception_handler(Exception)
async def global_exception_handler(request: Request, exc: Exception):
    logger.error(f"Unhandled error handling {request.method} {request.url.path}: {type(exc).__name__}")
    return JSONResponse(
        status_code=500,
        content={"error": {"message": "Internal gateway error", "type": "gateway_error", "code": 500}},
    )

@app.exception_handler(HTTPException)
async def http_exception_handler(request: Request, exc: HTTPException):
    return JSONResponse(
        status_code=exc.status_code,
        content={"error": {"message": exc.detail, "type": "http_error", "code": exc.status_code}},
    )

def auth(authorization: str | None) -> None:
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing or invalid authorization header")
    provided_token = authorization[7:].strip()
    if not secrets.compare_digest(provided_token, GATEWAY_TOKEN):
        raise HTTPException(401, "Unauthorized: invalid gateway token")

def internal_headers() -> dict[str, str]:
    return {
        "Authorization": f"Bearer {VLLM_API_KEY}",
        "Content-Type": "application/json",
    }

async def read_bounded_body(request: Request) -> bytes:
    content_length = request.headers.get("content-length")
    if content_length:
        try:
            declared = int(content_length)
        except ValueError:
            raise HTTPException(400, "Malformed Content-Length header")
        if declared > MAX_BODY_BYTES:
            raise HTTPException(413, f"Payload exceeds maximum allowed size ({MAX_BODY_BYTES} bytes)")

    body = bytearray()
    async for chunk in request.stream():
        body.extend(chunk)
        if len(body) > MAX_BODY_BYTES:
            raise HTTPException(413, f"Payload exceeds maximum allowed size ({MAX_BODY_BYTES} bytes)")
    return bytes(body)

@app.get("/healthz")
async def healthz(authorization: str | None = Header(default=None)):
    auth(authorization)
    req_id = secrets.token_hex(4)
    client = client_holder.get("client")
    if not client:
        return JSONResponse({"ok": False, "run_id": RUN_ID, "error": "Gateway starting"}, status_code=503)

    try:
        r = await client.get(f"{VLLM_URL}/v1/models", headers=internal_headers(), timeout=8.0)
        is_ready = r.status_code == 200
        logger.info(f"[{req_id}] /healthz -> upstream {r.status_code} (ready={is_ready})")
        return JSONResponse(
            {"ok": is_ready, "run_id": RUN_ID, "model": MODEL},
            status_code=200 if is_ready else 503,
        )
    except Exception as e:
        logger.warning(f"[{req_id}] /healthz check failed: {type(e).__name__}")
        return JSONResponse(
            {"ok": False, "run_id": RUN_ID, "error": "Upstream unavailable"},
            status_code=503,
        )

@app.get("/capabilities")
async def capabilities(authorization: str | None = Header(default=None)):
    auth(authorization)
    return {
        "runId": RUN_ID,
        "model": MODEL,
        "maxModelLen": MAX_MODEL_LEN,
        "multimodal": MULTIMODAL,
        "toolCalling": True,
        "streamUsage": True,
        "reasoningLevels": ["low", "medium", "xhigh"],
        "defaultReasoning": "xhigh",
        "backend": {
            "vllm_tpu": os.getenv("VLLM_TPU_VERSION", "0.28.0"),
            "mtp_tokens": MTP_TOKENS,
        },
    }

@app.get("/v1/models")
async def models(request: Request):
    auth(request.headers.get("authorization"))
    req_id = secrets.token_hex(4)
    client = client_holder.get("client")
    if not client:
        raise HTTPException(503, "Gateway unready")

    try:
        r = await client.get(f"{VLLM_URL}/v1/models", headers=internal_headers(), timeout=10.0)
        logger.info(f"[{req_id}] GET /v1/models -> {r.status_code}")
        try:
            content = r.json() if r.content else {}
        except Exception:
            content = {"error": {"message": f"Upstream returned non-JSON (status {r.status_code})", "type": "upstream_error", "code": r.status_code}}
        return JSONResponse(status_code=r.status_code, content=content)
    except httpx.RequestError:
        raise HTTPException(503, "Upstream inference service unavailable")

@app.post("/v1/chat/completions")
async def chat(request: Request):
    auth(request.headers.get("authorization"))
    req_id = secrets.token_hex(4)
    t0 = time.time()

    body = await read_bounded_body(request)

    try:
        payload = json.loads(body)
    except Exception:
        raise HTTPException(400, "Malformed JSON request body")
    if not isinstance(payload, dict):
        raise HTTPException(400, "Request body must be a JSON object")

    messages = payload.get("messages")
    if not isinstance(messages, list) or not messages:
        raise HTTPException(400, "'messages' must be a non-empty array")
    payload["model"] = MODEL
    
    # Handle both max_tokens and max_completion_tokens
    max_tokens = payload.get("max_tokens") or payload.get("max_completion_tokens")
    if max_tokens is not None:
        try:
            max_tokens = int(max_tokens)
        except (TypeError, ValueError):
            raise HTTPException(400, "'max_tokens' must be an integer")
        clamped_tokens = max(1, min(max_tokens, MAX_OUTPUT_TOKENS))
        payload["max_tokens"] = clamped_tokens
        if "max_completion_tokens" in payload:
            payload["max_completion_tokens"] = clamped_tokens

    stream_raw = payload.get("stream", False)
    stream = (
        stream_raw is True
        or stream_raw == 1
        or (isinstance(stream_raw, str) and stream_raw.lower() in ("1", "true"))
    )

    client = client_holder.get("client")
    if not client:
        raise HTTPException(503, "Gateway unready")

    body = json.dumps(payload).encode("utf-8")
    if len(body) > MAX_BODY_BYTES:
        raise HTTPException(413, f"Payload exceeds maximum allowed size ({MAX_BODY_BYTES} bytes)")

    logger.info(f"[{req_id}] POST /v1/chat/completions (stream={stream}, bytes={len(body)})")

    if not stream:
        try:
            await asyncio.wait_for(concurrency_limit.acquire(), timeout=30)
        except asyncio.TimeoutError:
            raise HTTPException(429, "Gateway busy: all inference slots occupied, retry shortly")

        async def call_upstream() -> tuple[int, dict]:
            try:
                r = await client.post(
                    f"{VLLM_URL}/v1/chat/completions",
                    content=body,
                    headers=internal_headers(),
                )
                logger.info(f"[{req_id}] Completed in {time.time() - t0:.2f}s (status={r.status_code})")
                try:
                    content = r.json() if r.content else {}
                except Exception:
                    content = {"error": {"message": f"Upstream returned non-JSON (status {r.status_code})", "type": "upstream_error", "code": r.status_code}}
                return r.status_code, content
            except httpx.RequestError:
                return 503, {"error": {"message": "Upstream inference service unavailable", "type": "http_error", "code": 503}}
            finally:
                concurrency_limit.release()

        task = asyncio.create_task(call_upstream())
        try:
            status, content = await asyncio.wait_for(asyncio.shield(task), timeout=NONSTREAM_WAIT_S)
            return JSONResponse(status_code=status, content=content)
        except asyncio.TimeoutError:
            pass

        # Cloudflare returns 524 when the origin sends nothing for 100 s. Past NONSTREAM_WAIT_S, send the
        # 200 header now and one space every 15 s (JSON allows leading whitespace), then the body.
        # Trade-off: an upstream error after this point arrives as an error JSON under status 200.
        logger.info(f"[{req_id}] Non-stream still running after {NONSTREAM_WAIT_S:.0f}s, keepalive padding on")

        async def padded() -> AsyncGenerator[bytes, None]:
            try:
                while not task.done():
                    done, _ = await asyncio.wait({task}, timeout=NONSTREAM_PAD_S)
                    if not done:
                        yield b" "
                yield json.dumps(task.result()[1]).encode("utf-8")
            finally:
                if not task.done():
                    task.cancel()
                    logger.info(f"[{req_id}] Downstream disconnected, aborting upstream call")

        return StreamingResponse(
            padded(),
            media_type="application/json",
            headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},
        )

    async def sse_generator() -> AsyncGenerator[bytes, None]:
        try:
            try:
                await asyncio.wait_for(concurrency_limit.acquire(), timeout=30)
            except asyncio.TimeoutError:
                err_payload = json.dumps({"error": {"message": "Gateway busy: all inference slots occupied, retry shortly", "type": "gateway_busy", "code": 429}})
                yield f"data: {err_payload}\n\ndata: [DONE]\n\n".encode("utf-8")
                return
            try:
                async with client.stream(
                    "POST",
                    f"{VLLM_URL}/v1/chat/completions",
                    content=body,
                    headers=internal_headers(),
                ) as r:
                    if r.status_code != 200:
                        error_bytes = await r.aread()
                        logger.warning(f"[{req_id}] Upstream error {r.status_code}")
                        try:
                            detail = json.loads(error_bytes.decode("utf-8")).get("error", {}).get(
                                "message", f"Upstream error {r.status_code}"
                            ) if error_bytes else f"Upstream error {r.status_code}"
                        except Exception:
                            detail = f"Upstream error {r.status_code}"
                        err_payload = json.dumps(
                            {"error": {"message": detail, "type": "upstream_error", "code": r.status_code}}
                        )
                        yield f"data: {err_payload}\n\ndata: [DONE]\n\n".encode("utf-8")
                        return

                    async for chunk in r.aiter_bytes():
                        if await request.is_disconnected():
                            logger.info(f"[{req_id}] Downstream disconnected, aborting upstream pull")
                            break
                        yield chunk
            finally:
                concurrency_limit.release()
        except httpx.RequestError as e:
            logger.error(f"[{req_id}] Upstream connection broken during streaming: {type(e).__name__}")
            err_payload = json.dumps({"error": {"message": "Upstream connection broken", "type": "upstream_disconnect"}})
            yield f"data: {err_payload}\n\ndata: [DONE]\n\n".encode("utf-8")
        finally:
            logger.info(f"[{req_id}] Stream closed after {time.time() - t0:.2f}s")

    return StreamingResponse(
        sse_generator(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "X-Accel-Buffering": "no",
            "Connection": "keep-alive",
        },
    )

In [ ]:
# 9) Gateway :8787 + tunnel -> in ENDPOINT. Chỉ gateway được đưa ra ngoài, máy chạy model thì không.
GW_TOKEN = GW_TOKEN_CO_DINH or ("gw-" + secrets.token_hex(20))
GW_FILE = Path("gateway.py").resolve()
assert GW_FILE.exists(), "Chưa chạy ô 8 (%%writefile gateway.py)."
for _ten in ("cf_proc", "gw_proc"):                                # chạy lại ô này: tắt bản cũ trước
    _p = globals().get(_ten)
    if _p is not None and _p.poll() is None:
        _p.terminate()
if ENGINE_PROC.poll() is not None:     # 13/10: bấm Stop ô 10 (bản notebook cũ) tắt luôn máy chạy model, gateway chờ vô ích
    raise SystemExit(f"{ENGINE_TEN} đã tắt (mã thoát {ENGINE_PROC.returncode}), gateway không có gì để nối. "
                     f"Chạy lại từ ô 6 (bấm vào ô 6 rồi Ctrl+F10). Log trong {LOG_DIR}")

gw_log = LOG_DIR / "gateway.log"
gw_proc = chay_nen(
    [sys.executable, "-m", "uvicorn", "gateway:app", "--host", "127.0.0.1", "--port", str(PORT_GW),
     "--workers", "1", "--log-level", "warning"],
    cwd=str(GW_FILE.parent), stdout=open(gw_log, "w"), stderr=subprocess.STDOUT,
    env=dict(os.environ, GATEWAY_TOKEN=GW_TOKEN, VLLM_API_KEY=API_KEY, VLLM_URL=f"http://127.0.0.1:{ENGINE_PORT}",
             SERVED_MODEL_NAME=ALIAS, MAX_MODEL_LEN=str(CTX_SIZE), TEXT_ONLY="false" if DOC_ANH else "true",
             GATEWAY_MAX_OUTPUT_TOKENS="32768", MTP_TOKENS=str(MTP_TOKENS), VLLM_TPU_VERSION=ENGINE_TEN,
             # số lượt gateway cho chạy cùng lúc = số lượt vLLM chạy song song (gateway.py mặc định 3: 13/10 G4 MAX_SEQS 4,
             # lượt thứ tư chờ rồi bị trả 429 "gateway busy")
             GATEWAY_CONCURRENCY=str(globals().get("MAX_SEQS", 3)),
             RUN_ID=str(int(time.time()))))
H_GW = {"Authorization": f"Bearer {GW_TOKEN}"}
LOCAL_GW = f"http://127.0.0.1:{PORT_GW}"
for _ in range(30):
    if gw_proc.poll() is not None:
        raise SystemExit("Gateway chết:\n" + gw_log.read_text(errors="ignore")[-3000:])
    try:
        if requests.get(LOCAL_GW + "/healthz", headers=H_GW, timeout=5).json().get("ok") is True:
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise SystemExit("Gateway không trả /healthz sau 30 giây:\n" + gw_log.read_text(errors="ignore")[-3000:])
print(f"gateway chạy ở 127.0.0.1:{PORT_GW} -> {ENGINE_TEN} 127.0.0.1:{ENGINE_PORT}")

MO_TUNNEL = (not ON_COLAB) if TUNNEL == "auto" else bool(TUNNEL)
BASE_URL, cf_proc = LOCAL_GW, None
if not MO_TUNNEL:
    print("KHÔNG mở tunnel: endpoint chỉ dùng được bên trong máy này.")
    if ON_COLAB:
        print('  Lý do mặc định tắt trên Colab: FAQ của Colab (đọc 13/10/2026) ghi "The following are disallowed from all managed\n'
              '  Colab runtimes", trong đó có "file hosting, media serving, or other web service offerings not related to interactive\n'
              '  compute with Colab" và "connecting to remote proxies" (https://research.google.com/colaboratory/faq.html).\n'
              "  Endpoint qua tunnel có thuộc diện đó không thì Colab không nói rõ; gói Pro không gỡ được danh sách này.\n"
              "  Anh tự quyết: đặt TUNNEL = True ở ô 2 rồi chạy lại ô 2 và ô 9.")
else:
    CF_VERSION = "2026.10.0"      # sha256 từ API GitHub ngày 13/10/2026
    CF_SHA256 = "d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db"
    cf_bin = BIN_DIR / f"cloudflared-{CF_VERSION}"
    if not cf_bin.exists():
        tai(f"https://github.com/cloudflare/cloudflared/releases/download/{CF_VERSION}/cloudflared-linux-amd64", cf_bin, CF_SHA256)
        os.chmod(cf_bin, 0o755)
    cf_log = LOG_DIR / "cloudflared.log"
    if CF_TOKEN:                  # tunnel có tên: địa chỉ cố định, cấu hình trong bảng điều khiển Cloudflare trỏ về http://127.0.0.1:8787
        cf_proc = chay_nen([str(cf_bin), "tunnel", "--no-autoupdate", "run"], stdout=open(cf_log, "w"),
                           stderr=subprocess.STDOUT, env=dict(os.environ, TUNNEL_TOKEN=CF_TOKEN))
        BASE_URL = PUBLIC_BASE_URL or None
        if not BASE_URL:
            print("Tunnel có tên đã chạy nhưng thiếu Secret PUBLIC_BASE_URL (ví dụ https://qwen.tenmien.com): không tự kiểm qua tunnel được.")
    else:                         # Quick Tunnel: không cần tài khoản, địa chỉ đổi mỗi phiên
        cf_proc = chay_nen([str(cf_bin), "tunnel", "--no-autoupdate", "--url", LOCAL_GW], stdout=open(cf_log, "w"),
                           stderr=subprocess.STDOUT)
        BASE_URL, _han = None, time.time() + 120
        while time.time() < _han and not BASE_URL and cf_proc.poll() is None:
            _tim = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", cf_log.read_text(errors="ignore"))
            BASE_URL = _tim.group(0) if _tim else None
            time.sleep(1)
        if not BASE_URL:
            raise SystemExit("Không mở được Quick Tunnel:\n" + cf_log.read_text(errors="ignore")[-3000:])
    if BASE_URL:                  # kiểm đường thật từ ngoài vào: /healthz rồi một lượt chat
        _ua = {**H_GW, "User-Agent": "Mozilla/5.0"}
        for _lan in range(40):
            try:
                if requests.get(BASE_URL + "/healthz", headers=_ua, timeout=10).json().get("ok") is True:
                    break
            except Exception:
                pass
            time.sleep(3)
        else:
            raise SystemExit(f"Gateway chưa trả lời qua tunnel sau 2 phút: {BASE_URL}")
        for _lan_chat in range(6):    # tunnel mới mở còn chập chờn: 13/10 /healthz đạt rồi lượt chat liền sau báo "Network is unreachable"
            try:
                j = hoi([{"role": "user", "content": "Reply with exactly: TUNNEL_OK"}], url=BASE_URL + "/v1/chat/completions",
                        headers=_ua, max_tokens=32, temperature=0, **KHONG_NGHI)
                break
            except requests.RequestException as e:
                _loi_chat = f"{type(e).__name__}: {str(e)[:200]}"
                time.sleep(5)
        else:
            raise SystemExit(f"/healthz qua tunnel đạt nhưng lượt chat hỏng 6 lần liền ({_loi_chat}). Chạy lại ô này.")
        assert "TUNNEL_OK" in (j["choices"][0]["message"].get("content") or ""), f"Chat qua tunnel trả sai: {json.dumps(j)[:300]}"
        print(f"[OK] Đã gọi được từ ngoài vào qua tunnel (thử /healthz lần {_lan + 1}, một lượt chat đúng).")

_url = BASE_URL or "<địa chỉ tunnel có tên của anh>"
_key = "<giá trị Secret GATEWAY_TOKEN của anh>" if GW_TOKEN_CO_DINH else GW_TOKEN     # khóa cố định thì không in ra output
print("\n" + "=" * 78)
print("  ENDPOINT SẴN SÀNG (chuẩn OpenAI)")
print("=" * 78)
print(f"  Base URL : {_url}/v1")
print(f"  API key  : {_key}")
print(f"  Model    : {ALIAS}")
_anh = "không" if not DOC_ANH else "có" if KET_QUA.get("đọc ảnh", "").startswith("đạt") else "có, nhưng ô 7 đọc SAI ảnh thử"
print(f"  Ngữ cảnh : {CTX_SIZE} token | đọc ảnh: {_anh} | nghĩ mặc định: {'bật' if THINKING else 'tắt'}"
      + (f" | {globals().get('SPEC_TEN') or 'MTP'} {MTP_TOKENS} token" if MTP_TOKENS else "")
      + (f" | code {TOC_DO:.1f} token/giây" if TOC_DO else ""))
print("-" * 78)
print("  Thử nhanh bằng curl:")
print(f"    curl {_url}/v1/chat/completions -H \"Authorization: Bearer {_key}\" -H \"Content-Type: application/json\" \\")
print(f"      -d '{{\"model\":\"{ALIAS}\",\"stream\":true,\"messages\":[{{\"role\":\"user\",\"content\":\"Xin chào\"}}]}}'")
print("  Python:")
print(f"    from openai import OpenAI; c = OpenAI(base_url=\"{_url}/v1\", api_key=\"<API key>\")")
print("-" * 78)
print("  Đường được mở: GET /healthz | GET /capabilities | GET /v1/models | POST /v1/chat/completions")
print("  Tắt nghĩ cho một lượt : thêm \"chat_template_kwargs\": {\"enable_thinking\": false}")
print("  Chỉnh mức nghĩ        : \"chat_template_kwargs\": {\"reasoning_effort\": \"low\" | \"medium\" | \"xhigh\"}")
print("  Nên gọi với \"stream\": true. Lượt không stream quá 60 giây: gateway gửi khoảng trắng giữ kết nối\n"
      "  (Cloudflare cắt lượt im lặng 100 giây), nên lỗi xảy ra sau mốc đó về dưới mã 200 kèm JSON lỗi.")
if MO_TUNNEL and not CF_TOKEN:
    print("  Địa chỉ trycloudflare.com và API key đổi mỗi phiên. Muốn cố định: xem bảng Secret ở đầu notebook.")
print("=" * 78)

In [ ]:
# 10) Giữ phiên sống và báo nhịp. Bấm Stop ở ô này KHÔNG tắt endpoint; muốn tắt hẳn thì chạy tat_het().
def tat_het():
    for ten, p in (("tunnel", cf_proc), ("gateway", gw_proc), (ENGINE_TEN, ENGINE_PROC)):
        if p is not None and p.poll() is None:
            p.terminate()
            print("đã tắt", ten)


def gio_vn():
    return time.strftime("%H:%M", time.gmtime(time.time() + 7 * 3600))


_het = time.time() + KEEP_ALIVE_HOURS * 3600
print(f"{gio_vn()} (giờ VN): giữ endpoint {KEEP_ALIVE_HOURS} giờ rồi tự tắt. Sửa KEEP_ALIVE_HOURS ở ô 2 nếu cần.")
try:
    while time.time() < _het:
        _chet = [t for t, p in ((ENGINE_TEN, ENGINE_PROC), ("gateway", gw_proc), ("tunnel", cf_proc))
                 if p is not None and p.poll() is not None]
        if _chet:
            print(f"{gio_vn()} ĐÃ CHẾT: {_chet}. Xem log trong {LOG_DIR}, rồi chạy lại từ ô 6.")
            break
        _luot = gw_log.read_text(errors="ignore").count("POST /v1/chat/completions")
        _gpu = " ".join(f"GPU{i}:{u}MiB/{p_}%" for i, u, p_ in gpu_csv("index,memory.used,utilization.gpu"))
        print(f"{gio_vn()} | còn {(_het - time.time()) / 3600:.1f} giờ | {_luot} lượt chat | {_gpu}", flush=True)
        time.sleep(min(300, max(1, _het - time.time())))
    else:
        print(f"{gio_vn()} hết {KEEP_ALIVE_HOURS} giờ, tắt endpoint.")
        tat_het()
except KeyboardInterrupt:
    print("Đã dừng ô báo nhịp. Endpoint VẪN chạy. Tắt hẳn: chạy tat_het() trong một ô mới.")